# 00 — Descarga de Datos: Home Credit Default Risk

Este notebook descarga los datos de la competencia de Kaggle y realiza una inspección inicial de cada tabla.

## Prerrequisitos

1. **Cuenta Kaggle**: crear en [kaggle.com](https://www.kaggle.com)
2. **Aceptar términos**: ir a la [competencia](https://www.kaggle.com/competitions/home-credit-default-risk) → pestaña *Rules* → *I Understand and Accept*
3. **API Token**: `Account → API → Create New API Token` → descarga `kaggle.json`
4. **Guardar credenciales**: mover `kaggle.json` a `~/.kaggle/kaggle.json`
   - En Windows: `C:\Users\<tu_usuario>\.kaggle\kaggle.json`
   - En Linux/Mac: `~/.kaggle/kaggle.json`
5. **Instalar librería**: `pip install kaggle`

## Datasets que se descargarán

| Archivo | Filas aprox. | Descripción |
|---------|-------------|-------------|
| `application_train.csv` | 307k | Solicitudes de crédito con TARGET (0=paga, 1=default) |
| `application_test.csv` | 49k | Solicitudes para predicción (sin TARGET) |
| `bureau.csv` | 1.7M | Historial crediticio en otras instituciones (bureau) |
| `bureau_balance.csv` | 27M | Saldo mensual de créditos en bureau |
| `previous_application.csv` | 1.7M | Solicitudes previas en Home Credit |
| `POS_CASH_balance.csv` | 10M | Saldo mensual de créditos POS y efectivo previos |
| `credit_card_balance.csv` | 3.8M | Snapshots mensuales de tarjetas de crédito previas |
| `installments_payments.csv` | 13.6M | Historial de pagos de créditos previos |
| `HomeCredit_columns_description.csv` | — | Diccionario de variables |

## Relaciones entre Tablas

```
application_train.csv  ←── Tabla principal (SK_ID_CURR)
application_test.csv
        │
        ├─── bureau.csv  (SK_ID_CURR)
        │         └─── bureau_balance.csv  (SK_ID_BUREAU)
        │
        └─── previous_application.csv  (SK_ID_CURR)
                    ├─── POS_CASH_balance.csv      (SK_ID_PREV)
                    ├─── credit_card_balance.csv    (SK_ID_PREV)
                    └─── installments_payments.csv  (SK_ID_PREV)
```

La estrategia de feature engineering consistirá en **agregar** las tablas secundarias a nivel `SK_ID_CURR` (y en algunos casos `SK_ID_PREV`) para luego unirlas con la tabla principal.

In [5]:
import os
import sys

# --- Verificar credenciales de Kaggle ---
kaggle_path = os.path.join(os.path.expanduser('~'), '.kaggle', 'kaggle.json')
if not os.path.exists(kaggle_path):
    raise FileNotFoundError(
        f'No se encontró kaggle.json en {kaggle_path}\n'
        'Ver instrucciones al inicio de este notebook.'
    )
# Forzar permisos correctos en Linux/Mac
if sys.platform != 'win32':
    os.chmod(kaggle_path, 0o600)
print(f'Credenciales OK: {kaggle_path}')

Credenciales OK: C:\Users\HP\.kaggle\kaggle.json


In [6]:
import kaggle

DATA_RAW = os.path.join('..', 'data', 'raw')
os.makedirs(DATA_RAW, exist_ok=True)
print(f'Descargando datos en: {os.path.abspath(DATA_RAW)}')

kaggle.api.authenticate()
kaggle.api.competition_download_files(
    'home-credit-default-risk',
    path=DATA_RAW,
    quiet=False
)
print('Descarga completada.')

Descargando datos en: c:\Users\HP\OneDrive\Escritorio\David Guzzi\Github\MECMT07\data\raw
home-credit-default-risk.zip: Skipping, found more recently modified local copy (use --force to force download)
Descarga completada.


In [7]:
import zipfile
import glob

zips = glob.glob(os.path.join(DATA_RAW, '*.zip'))
print(f'Archivos zip encontrados: {len(zips)}')

for zf_path in zips:
    print(f'  Extrayendo: {os.path.basename(zf_path)} ...')
    with zipfile.ZipFile(zf_path, 'r') as zf:
        zf.extractall(DATA_RAW)
print('Extracción completa.')

Archivos zip encontrados: 1
  Extrayendo: home-credit-default-risk.zip ...
Extracción completa.


In [8]:
# Listar archivos resultantes
files = sorted(glob.glob(os.path.join(DATA_RAW, '*.csv')))
print(f'CSVs disponibles ({len(files)}):')
for f in files:
    size_mb = os.path.getsize(f) / 1e6
    print(f'  {os.path.basename(f):50s}  {size_mb:8.1f} MB')

CSVs disponibles (10):
  HomeCredit_columns_description.csv                       0.0 MB
  POS_CASH_balance.csv                                   392.7 MB
  application_test.csv                                    26.6 MB
  application_train.csv                                  166.1 MB
  bureau.csv                                             170.0 MB
  bureau_balance.csv                                     375.6 MB
  credit_card_balance.csv                                424.6 MB
  installments_payments.csv                              723.1 MB
  previous_application.csv                               405.0 MB
  sample_submission.csv                                    0.5 MB


## Inspección de Cada Dataset

Para cada tabla se reporta: shape, tipos de variables, valores faltantes y primeras filas.

In [9]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 120)

def inspect(filepath, target_col=None, nrows=None):
    """Inspección rápida de un CSV."""
    name = os.path.basename(filepath)
    df = pd.read_csv(filepath, nrows=nrows)
    print(f'{'='*70}')
    print(f'  {name}')
    print(f'{'='*70}')
    print(f'  Shape        : {df.shape[0]:,} filas × {df.shape[1]} columnas')
    
    # Tipos
    dtypes = df.dtypes.value_counts()
    print(f'  Tipos        : {dict(dtypes)}')
    
    # Missings
    miss = df.isnull().sum()
    miss_pct = (miss / len(df) * 100).round(1)
    miss_summary = miss[miss > 0].sort_values(ascending=False)
    print(f'  Cols con NA  : {len(miss_summary)} / {df.shape[1]}')
    if len(miss_summary) > 0:
        top5 = miss_summary.head(5)
        for col, cnt in top5.items():
            print(f'    {col:40s} {cnt:>8,} ({miss_pct[col]:.1f}%)')
    
    # Target (si aplica)
    if target_col and target_col in df.columns:
        vc = df[target_col].value_counts()
        pct1 = vc.get(1, 0) / len(df) * 100
        print(f'  TARGET=0     : {vc.get(0, 0):,}  ({100-pct1:.1f}%)')
        print(f'  TARGET=1     : {vc.get(1, 0):,}  ({pct1:.1f}%)  ← default')
    
    print()
    display(df.head(2))
    print()

In [10]:
inspect(os.path.join(DATA_RAW, 'application_train.csv'), target_col='TARGET')

  application_train.csv
  Shape        : 307,511 filas × 122 columnas
  Tipos        : {dtype('float64'): np.int64(65), dtype('int64'): np.int64(41), dtype('O'): np.int64(16)}
  Cols con NA  : 67 / 122
    COMMONAREA_MEDI                           214,865 (69.9%)
    COMMONAREA_MODE                           214,865 (69.9%)
    COMMONAREA_AVG                            214,865 (69.9%)
    NONLIVINGAPARTMENTS_MODE                  213,514 (69.4%)
    NONLIVINGAPARTMENTS_MEDI                  213,514 (69.4%)
  TARGET=0     : 282,686  (91.9%)
  TARGET=1     : 24,825  (8.1%)  ← default



,SK_ID_CURR,TARGET,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100002,1,Cash loans,M,N,Y,0,202500.0,406597.5,24700.5,351000.0,Unaccompanied,Working,Secondary / secondary special,Single / not married,House / apartment,0.018801,-9461,-637,-3648.0,-2120,NaN,1,1,0,1,1,0,Laborers,1.0,2,2,WEDNESDAY,10,0,0,0,0,0,0,Business Entity Type 3,0.083037,0.262949,0.139376,0.0247,0.0369,0.9722,0.6192,0.0143,0.00,0.0690,0.0833,0.1250,0.0369,0.0202,0.0190,0.0000,0.0000,0.0252,0.0383,0.9722,0.6341,0.0144,0.0000,0.0690,0.0833,0.1250,0.0377,0.022,0.0198,0.0,0.0,0.0250,0.0369,0.9722,0.6243,0.0144,0.00,0.0690,0.0833,0.1250,0.0375,0.0205,0.0193,0.0000,0.00,reg oper account,block of flats,0.0149,"Stone, brick",No,2.0,2.0,2.0,2.0,-1134.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,1.0
1,100003,0,Cash loans,F,N,N,0,270000.0,1293502.5,35698.5,1129500.0,Family,State servant,Higher education,Married,House / apartment,0.003541,-16765,-1188,-1186.0,-291,NaN,1,1,0,1,1,0,Core staff,2.0,1,1,MONDAY,11,0,0,0,0,0,0,School,0.311267,0.622246,NaN,0.0959,0.0529,0.9851,0.7960,0.0605,0.08,0.0345,0.2917,0.3333,0.0130,0.0773,0.0549,0.0039,0.0098,0.0924,0.0538,0.9851,0.8040,0.0497,0.0806,0.0345,0.2917,0.3333,0.0128,0.079,0.0554,0.0,0.0,0.0968,0.0529,0.9851,0.7987,0.0608,0.08,0.0345,0.2917,0.3333,0.0132,0.0787,0.0558,0.0039,0.01,reg oper account,block of flats,0.0714,Block,No,1.0,0.0,1.0,0.0,-828.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0


In [11]:
inspect(os.path.join(DATA_RAW, 'application_test.csv'))

  application_test.csv
  Shape        : 48,744 filas × 121 columnas
  Tipos        : {dtype('float64'): np.int64(65), dtype('int64'): np.int64(40), dtype('O'): np.int64(16)}
  Cols con NA  : 64 / 121
    COMMONAREA_AVG                             33,495 (68.7%)
    COMMONAREA_MODE                            33,495 (68.7%)
    COMMONAREA_MEDI                            33,495 (68.7%)
    NONLIVINGAPARTMENTS_AVG                    33,347 (68.4%)
    NONLIVINGAPARTMENTS_MODE                   33,347 (68.4%)



,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,NAME_TYPE_SUITE,NAME_INCOME_TYPE,NAME_EDUCATION_TYPE,NAME_FAMILY_STATUS,NAME_HOUSING_TYPE,REGION_POPULATION_RELATIVE,DAYS_BIRTH,DAYS_EMPLOYED,DAYS_REGISTRATION,DAYS_ID_PUBLISH,OWN_CAR_AGE,FLAG_MOBIL,FLAG_EMP_PHONE,FLAG_WORK_PHONE,FLAG_CONT_MOBILE,FLAG_PHONE,FLAG_EMAIL,OCCUPATION_TYPE,CNT_FAM_MEMBERS,REGION_RATING_CLIENT,REGION_RATING_CLIENT_W_CITY,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,REG_REGION_NOT_LIVE_REGION,REG_REGION_NOT_WORK_REGION,LIVE_REGION_NOT_WORK_REGION,REG_CITY_NOT_LIVE_CITY,REG_CITY_NOT_WORK_CITY,LIVE_CITY_NOT_WORK_CITY,ORGANIZATION_TYPE,EXT_SOURCE_1,EXT_SOURCE_2,EXT_SOURCE_3,APARTMENTS_AVG,BASEMENTAREA_AVG,YEARS_BEGINEXPLUATATION_AVG,YEARS_BUILD_AVG,COMMONAREA_AVG,ELEVATORS_AVG,ENTRANCES_AVG,FLOORSMAX_AVG,FLOORSMIN_AVG,LANDAREA_AVG,LIVINGAPARTMENTS_AVG,LIVINGAREA_AVG,NONLIVINGAPARTMENTS_AVG,NONLIVINGAREA_AVG,APARTMENTS_MODE,BASEMENTAREA_MODE,YEARS_BEGINEXPLUATATION_MODE,YEARS_BUILD_MODE,COMMONAREA_MODE,ELEVATORS_MODE,ENTRANCES_MODE,FLOORSMAX_MODE,FLOORSMIN_MODE,LANDAREA_MODE,LIVINGAPARTMENTS_MODE,LIVINGAREA_MODE,NONLIVINGAPARTMENTS_MODE,NONLIVINGAREA_MODE,APARTMENTS_MEDI,BASEMENTAREA_MEDI,YEARS_BEGINEXPLUATATION_MEDI,YEARS_BUILD_MEDI,COMMONAREA_MEDI,ELEVATORS_MEDI,ENTRANCES_MEDI,FLOORSMAX_MEDI,FLOORSMIN_MEDI,LANDAREA_MEDI,LIVINGAPARTMENTS_MEDI,LIVINGAREA_MEDI,NONLIVINGAPARTMENTS_MEDI,NONLIVINGAREA_MEDI,FONDKAPREMONT_MODE,HOUSETYPE_MODE,TOTALAREA_MODE,WALLSMATERIAL_MODE,EMERGENCYSTATE_MODE,OBS_30_CNT_SOCIAL_CIRCLE,DEF_30_CNT_SOCIAL_CIRCLE,OBS_60_CNT_SOCIAL_CIRCLE,DEF_60_CNT_SOCIAL_CIRCLE,DAYS_LAST_PHONE_CHANGE,FLAG_DOCUMENT_2,FLAG_DOCUMENT_3,FLAG_DOCUMENT_4,FLAG_DOCUMENT_5,FLAG_DOCUMENT_6,FLAG_DOCUMENT_7,FLAG_DOCUMENT_8,FLAG_DOCUMENT_9,FLAG_DOCUMENT_10,FLAG_DOCUMENT_11,FLAG_DOCUMENT_12,FLAG_DOCUMENT_13,FLAG_DOCUMENT_14,FLAG_DOCUMENT_15,FLAG_DOCUMENT_16,FLAG_DOCUMENT_17,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,Unaccompanied,Working,Higher education,Married,House / apartment,0.018850,-19241,-2329,-5170.0,-812,NaN,1,1,0,1,0,1,NaN,2.0,2,2,TUESDAY,18,0,0,0,0,0,0,Kindergarten,0.752614,0.789654,0.159520,0.066,0.059,0.9732,NaN,NaN,NaN,0.1379,0.125,NaN,NaN,NaN,0.0505,NaN,NaN,0.0672,0.0612,0.9732,NaN,NaN,NaN,0.1379,0.125,NaN,NaN,NaN,0.0526,NaN,NaN,0.0666,0.059,0.9732,NaN,NaN,NaN,0.1379,0.125,NaN,NaN,NaN,0.0514,NaN,NaN,NaN,block of flats,0.0392,"Stone, brick",No,0.0,0.0,0.0,0.0,-1740.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,Unaccompanied,Working,Secondary / secondary special,Married,House / apartment,0.035792,-18064,-4469,-9118.0,-1623,NaN,1,1,0,1,0,0,Low-skill Laborers,2.0,2,2,FRIDAY,9,0,0,0,0,0,0,Self-employed,0.564990,0.291656,0.432962,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.0,0.0,0.0,0.0,0.0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0


In [12]:
inspect(os.path.join(DATA_RAW, 'bureau.csv'))

  bureau.csv
  Shape        : 1,716,428 filas × 17 columnas
  Tipos        : {dtype('float64'): np.int64(8), dtype('int64'): np.int64(6), dtype('O'): np.int64(3)}
  Cols con NA  : 7 / 17
    AMT_ANNUITY                              1,226,791 (71.5%)
    AMT_CREDIT_MAX_OVERDUE                   1,124,488 (65.5%)
    DAYS_ENDDATE_FACT                         633,653 (36.9%)
    AMT_CREDIT_SUM_LIMIT                      591,780 (34.5%)
    AMT_CREDIT_SUM_DEBT                       257,669 (15.0%)



,SK_ID_CURR,SK_ID_BUREAU,CREDIT_ACTIVE,CREDIT_CURRENCY,DAYS_CREDIT,CREDIT_DAY_OVERDUE,DAYS_CREDIT_ENDDATE,DAYS_ENDDATE_FACT,AMT_CREDIT_MAX_OVERDUE,CNT_CREDIT_PROLONG,AMT_CREDIT_SUM,AMT_CREDIT_SUM_DEBT,AMT_CREDIT_SUM_LIMIT,AMT_CREDIT_SUM_OVERDUE,CREDIT_TYPE,DAYS_CREDIT_UPDATE,AMT_ANNUITY
0,215354,5714462,Closed,currency 1,-497,0,-153.0,-153.0,NaN,0,91323.0,0.0,NaN,0.0,Consumer credit,-131,NaN
1,215354,5714463,Active,currency 1,-208,0,1075.0,NaN,NaN,0,225000.0,171342.0,NaN,0.0,Credit card,-20,NaN


In [13]:
# bureau_balance es muy grande — leer solo primeras 100k filas para la inspección
inspect(os.path.join(DATA_RAW, 'bureau_balance.csv'), nrows=100_000)

  bureau_balance.csv
  Shape        : 100,000 filas × 3 columnas
  Tipos        : {dtype('int64'): np.int64(2), dtype('O'): np.int64(1)}
  Cols con NA  : 0 / 3



,SK_ID_BUREAU,MONTHS_BALANCE,STATUS
0,5715448,0,C
1,5715448,-1,C


In [14]:
inspect(os.path.join(DATA_RAW, 'previous_application.csv'))

  previous_application.csv
  Shape        : 1,670,214 filas × 37 columnas
  Tipos        : {dtype('O'): np.int64(16), dtype('float64'): np.int64(15), dtype('int64'): np.int64(6)}
  Cols con NA  : 16 / 37
    RATE_INTEREST_PRIVILEGED                 1,664,263 (99.6%)
    RATE_INTEREST_PRIMARY                    1,664,263 (99.6%)
    AMT_DOWN_PAYMENT                          895,844 (53.6%)
    RATE_DOWN_PAYMENT                         895,844 (53.6%)
    NAME_TYPE_SUITE                           820,405 (49.1%)



,SK_ID_PREV,SK_ID_CURR,NAME_CONTRACT_TYPE,AMT_ANNUITY,AMT_APPLICATION,AMT_CREDIT,AMT_DOWN_PAYMENT,AMT_GOODS_PRICE,WEEKDAY_APPR_PROCESS_START,HOUR_APPR_PROCESS_START,FLAG_LAST_APPL_PER_CONTRACT,NFLAG_LAST_APPL_IN_DAY,RATE_DOWN_PAYMENT,RATE_INTEREST_PRIMARY,RATE_INTEREST_PRIVILEGED,NAME_CASH_LOAN_PURPOSE,NAME_CONTRACT_STATUS,DAYS_DECISION,NAME_PAYMENT_TYPE,CODE_REJECT_REASON,NAME_TYPE_SUITE,NAME_CLIENT_TYPE,NAME_GOODS_CATEGORY,NAME_PORTFOLIO,NAME_PRODUCT_TYPE,CHANNEL_TYPE,SELLERPLACE_AREA,NAME_SELLER_INDUSTRY,CNT_PAYMENT,NAME_YIELD_GROUP,PRODUCT_COMBINATION,DAYS_FIRST_DRAWING,DAYS_FIRST_DUE,DAYS_LAST_DUE_1ST_VERSION,DAYS_LAST_DUE,DAYS_TERMINATION,NFLAG_INSURED_ON_APPROVAL
0,2030495,271877,Consumer loans,1730.430,17145.0,17145.0,0.0,17145.0,SATURDAY,15,Y,1,0.0,0.182832,0.867336,XAP,Approved,-73,Cash through the bank,XAP,NaN,Repeater,Mobile,POS,XNA,Country-wide,35,Connectivity,12.0,middle,POS mobile with interest,365243.0,-42.0,300.0,-42.0,-37.0,0.0
1,2802425,108129,Cash loans,25188.615,607500.0,679671.0,NaN,607500.0,THURSDAY,11,Y,1,NaN,NaN,NaN,XNA,Approved,-164,XNA,XAP,Unaccompanied,Repeater,XNA,Cash,x-sell,Contact center,-1,XNA,36.0,low_action,Cash X-Sell: low,365243.0,-134.0,916.0,365243.0,365243.0,1.0


In [15]:
inspect(os.path.join(DATA_RAW, 'POS_CASH_balance.csv'), nrows=100_000)

  POS_CASH_balance.csv
  Shape        : 100,000 filas × 8 columnas
  Tipos        : {dtype('int64'): np.int64(5), dtype('float64'): np.int64(2), dtype('O'): np.int64(1)}
  Cols con NA  : 2 / 8
    CNT_INSTALMENT                                245 (0.2%)
    CNT_INSTALMENT_FUTURE                         245 (0.2%)



,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,CNT_INSTALMENT,CNT_INSTALMENT_FUTURE,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,1803195,182943,-31,48.0,45.0,Active,0,0
1,1715348,367990,-33,36.0,35.0,Active,0,0


In [16]:
inspect(os.path.join(DATA_RAW, 'credit_card_balance.csv'), nrows=100_000)

  credit_card_balance.csv
  Shape        : 100,000 filas × 23 columnas
  Tipos        : {dtype('float64'): np.int64(15), dtype('int64'): np.int64(7), dtype('O'): np.int64(1)}
  Cols con NA  : 9 / 23
    AMT_DRAWINGS_ATM_CURRENT                   22,233 (22.2%)
    AMT_DRAWINGS_OTHER_CURRENT                 22,233 (22.2%)
    AMT_DRAWINGS_POS_CURRENT                   22,233 (22.2%)
    CNT_DRAWINGS_OTHER_CURRENT                 22,233 (22.2%)
    CNT_DRAWINGS_ATM_CURRENT                   22,233 (22.2%)



,SK_ID_PREV,SK_ID_CURR,MONTHS_BALANCE,AMT_BALANCE,AMT_CREDIT_LIMIT_ACTUAL,AMT_DRAWINGS_ATM_CURRENT,AMT_DRAWINGS_CURRENT,AMT_DRAWINGS_OTHER_CURRENT,AMT_DRAWINGS_POS_CURRENT,AMT_INST_MIN_REGULARITY,AMT_PAYMENT_CURRENT,AMT_PAYMENT_TOTAL_CURRENT,AMT_RECEIVABLE_PRINCIPAL,AMT_RECIVABLE,AMT_TOTAL_RECEIVABLE,CNT_DRAWINGS_ATM_CURRENT,CNT_DRAWINGS_CURRENT,CNT_DRAWINGS_OTHER_CURRENT,CNT_DRAWINGS_POS_CURRENT,CNT_INSTALMENT_MATURE_CUM,NAME_CONTRACT_STATUS,SK_DPD,SK_DPD_DEF
0,2562384,378907,-6,56.970,135000,0.0,877.5,0.0,877.5,1700.325,1800.0,1800.0,0.00,0.000,0.000,0.0,1,0.0,1.0,35.0,Active,0,0
1,2582071,363914,-1,63975.555,45000,2250.0,2250.0,0.0,0.0,2250.000,2250.0,2250.0,60175.08,64875.555,64875.555,1.0,1,0.0,0.0,69.0,Active,0,0


In [17]:
inspect(os.path.join(DATA_RAW, 'installments_payments.csv'), nrows=100_000)

  installments_payments.csv
  Shape        : 100,000 filas × 8 columnas
  Tipos        : {dtype('float64'): np.int64(5), dtype('int64'): np.int64(3)}
  Cols con NA  : 0 / 8



,SK_ID_PREV,SK_ID_CURR,NUM_INSTALMENT_VERSION,NUM_INSTALMENT_NUMBER,DAYS_INSTALMENT,DAYS_ENTRY_PAYMENT,AMT_INSTALMENT,AMT_PAYMENT
0,1054186,161674,1.0,6,-1180.0,-1187.0,6948.360,6948.360
1,1330831,151639,0.0,34,-2156.0,-2156.0,1716.525,1716.525


In [18]:
# Diccionario de variables
desc = pd.read_csv(os.path.join(DATA_RAW, 'HomeCredit_columns_description.csv'), encoding='latin1')
print(f'Diccionario: {desc.shape[0]} variables documentadas')
display(desc.head(10))

Diccionario: 219 variables documentadas


,Unnamed: 0,Table,Row,Description,Special
0,1,application_{train|test}.csv,SK_ID_CURR,ID of loan in our sample,NaN
1,2,application_{train|test}.csv,TARGET,Target variable (1 - client with payment diffi...,NaN
2,5,application_{train|test}.csv,NAME_CONTRACT_TYPE,Identification if loan is cash or revolving,NaN
3,6,application_{train|test}.csv,CODE_GENDER,Gender of the client,NaN
4,7,application_{train|test}.csv,FLAG_OWN_CAR,Flag if the client owns a car,NaN
5,8,application_{train|test}.csv,FLAG_OWN_REALTY,Flag if client owns a house or flat,NaN
6,9,application_{train|test}.csv,CNT_CHILDREN,Number of children the client has,NaN
7,10,application_{train|test}.csv,AMT_INCOME_TOTAL,Income of the client,NaN
8,11,application_{train|test}.csv,AMT_CREDIT,Credit amount of the loan,NaN
9,12,application_{train|test}.csv,AMT_ANNUITY,Loan annuity,NaN


## Resumen

Los datos están listos en `data/raw/`. La siguiente etapa es el análisis exploratorio detallado:
- `01_eda_application.ipynb` — tabla principal (train + test)
- `02_eda_bureau.ipynb` — bureau y bureau_balance
- `03_eda_previous_applications.ipynb` — solicitudes previas
- `04_eda_balance_tables.ipynb` — POS_CASH, credit_card, installments